In [1]:
import pandas as pd

# Paths to your final datasets (adjust filenames as needed)
train_path = "Node A_train.csv"
test_path  = "Node A_test.csv"

# Load datasets
df_train = pd.read_csv(train_path)
df_test  = pd.read_csv(test_path)

print("Final mixed train shape:", df_train.shape)
print("Final mixed test shape: ", df_test.shape)
print("\nColumns:", df_train.columns.tolist())
print("\nTarget value counts (train):")
print(df_train["Attack"].value_counts())
print("\nTarget value counts (test):")
print(df_test["Attack"].value_counts())

Final mixed train shape: (30732, 7)
Final mixed test shape:  (13182, 7)

Columns: ['shunt_voltage', 'bus_voltage_V', 'current_mA', 'power_mW', 'State', 'Attack', 'data_source']

Target value counts (train):
Attack
none         13002
syn-flood    10638
Backdoor      7092
Name: count, dtype: int64

Target value counts (test):
Attack
none         5577
syn-flood    4563
Backdoor     3042
Name: count, dtype: int64


In [3]:
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, log_loss, mean_squared_error, roc_auc_score

# =============================================================================
# 1. Prepare features and target
# =============================================================================
target_col = "Attack"
feature_cols = ["shunt_voltage", "bus_voltage_V", "current_mA", "power_mW", "State"]

# Separate features and target
X_train_raw = df_train[feature_cols].copy()
y_train_raw = df_train[target_col].copy()

X_test_raw  = df_test[feature_cols].copy()
y_test_raw  = df_test[target_col].copy()

# One-hot encode 'State'
X_train = pd.get_dummies(X_train_raw, columns=["State"], drop_first=False)
X_test  = pd.get_dummies(X_test_raw,  columns=["State"], drop_first=False)

# Ensure same columns in train and test (in case some State value is missing in one split)
for col in ["State_idle", "State_charging"]:
    if col not in X_train.columns:
        X_train[col] = 0
    if col not in X_test.columns:
        X_test[col] = 0

X_train = X_train[["shunt_voltage", "bus_voltage_V", "current_mA", "power_mW", "State_idle", "State_charging"]]
X_test  = X_test[["shunt_voltage", "bus_voltage_V", "current_mA", "power_mW", "State_idle", "State_charging"]]

# Encode target labels
le = LabelEncoder()
y_train = le.fit_transform(y_train_raw)
y_test  = le.transform(y_test_raw)

print("Encoded classes:", dict(zip(le.classes_, range(len(le.classes_)))))
print("Feature columns:", X_train.columns.tolist())

# =============================================================================
# 2. Scale numeric features (StandardScaler on numeric part only)
# =============================================================================
from sklearn.preprocessing import StandardScaler

numeric_features = ["shunt_voltage", "bus_voltage_V", "current_mA", "power_mW"]

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled  = X_test.copy()

X_train_scaled[numeric_features] = scaler.fit_transform(X_train[numeric_features])
X_test_scaled[numeric_features]  = scaler.transform(X_test[numeric_features])

# Convert to numpy for sklearn
X_train_np = X_train_scaled.to_numpy()
X_test_np  = X_test_scaled.to_numpy()

# =============================================================================
# 3. Train Random Forest with your chosen hyperparameters
# =============================================================================
rf_final = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    oob_score=True,
    n_jobs=-1
)

rf_final.fit(X_train_np, y_train)

print("\nRandom Forest trained on final mixed dataset.")
print(f"Training samples: {X_train_np.shape[0]}, Features: {X_train_np.shape[1]}")
print(f"OOB score (train): {rf_final.oob_score_:.6f}")

Encoded classes: {'Backdoor': 0, 'none': 1, 'syn-flood': 2}
Feature columns: ['shunt_voltage', 'bus_voltage_V', 'current_mA', 'power_mW', 'State_idle', 'State_charging']

Random Forest trained on final mixed dataset.
Training samples: 30732, Features: 6
OOB score (train): 0.868834


In [4]:
from sklearn.metrics import f1_score, log_loss, mean_squared_error, roc_auc_score, confusion_matrix

# Predictions
y_test_pred = rf_final.predict(X_test_np)
y_test_prob = rf_final.predict_proba(X_test_np)

# Macro F1
macro_f1 = f1_score(y_test, y_test_pred, average="macro")

# Reconstruction error (MSE between true and predicted labels)
reconstruction_error = mean_squared_error(y_test, y_test_pred)

# Losses
test_loss = log_loss(y_test, y_test_prob)

# OOB validation loss (from training)
val_loss = None
if hasattr(rf_final, "oob_decision_function_") and rf_final.oob_decision_function_ is not None:
    val_loss = log_loss(y_train, rf_final.oob_decision_function_)

# ROC-AUC (macro, one-vs-rest)
roc_auc = roc_auc_score(y_test, y_test_prob, multi_class="ovr", average="macro")

print("==========================================")
print(" Evaluation Metrics: Final Mixed Dataset  ")
print("==========================================")
print(f"  Macro_f1:             {macro_f1:.6f}")
print(f"  Reconstruction_Error: {reconstruction_error:.6f}")
print(f"  test_loss:            {test_loss:.6f}")
if val_loss is not None:
    print(f"  validation_loss:      {val_loss:.6f}")
else:
    print("  validation_loss:      Not available (oob_decision_function_ is None)")
print(f"  ROC_AUC:              {roc_auc:.6f}")

# Confusion matrix
cm = confusion_matrix(y_test, y_test_pred)
print("\nConfusion Matrix (rows: true, cols: predicted):")
print("Classes:", le.classes_)
print(cm)

 Evaluation Metrics: Final Mixed Dataset  
  Macro_f1:             0.855424
  Reconstruction_Error: 0.126384
  test_loss:            0.278408
  validation_loss:      0.313319
  ROC_AUC:              0.959268

Confusion Matrix (rows: true, cols: predicted):
Classes: ['Backdoor' 'none' 'syn-flood']
[[2021 1017    4]
 [ 630 4946    1]
 [   0    2 4561]]


In [5]:
# perfect mixture between all 3 models and now the mixed is almost every time in the middle with the results

In [6]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, log_loss, mean_squared_error, roc_auc_score

# =============================================================================
# 1. LOAD NON-NORMALIZED DATASETS
# =============================================================================
train_path = "Node A_train.csv"
test_path  = "Node A_test.csv"

df_train = pd.read_csv(train_path)
df_test  = pd.read_csv(test_path)

print("=== Non-normalized datasets ===")
print("Train shape:", df_train.shape)
print("Test shape: ", df_test.shape)


=== Non-normalized datasets ===
Train shape: (30732, 7)
Test shape:  (13182, 7)


In [7]:
# =============================================================================
# 2. LOAD NORMALIZED DATASETS
# =============================================================================
norm_train_path = "dataset/normalized/Node_A_train_normalized.csv"
norm_test_path  = "dataset/normalized/Node_A_test_normalized.csv"

df_train_norm = pd.read_csv(norm_train_path)
df_test_norm  = pd.read_csv(norm_test_path)

print("\n=== Normalized datasets ===")
print("Train shape:", df_train_norm.shape)
print("Test shape: ", df_test_norm.shape)


=== Normalized datasets ===
Train shape: (30732, 7)
Test shape:  (13182, 7)


In [8]:
# =============================================================================
# 3. COMMON PREPROCESSING: FEATURES, TARGET, ONE-HOT ENCODING
# =============================================================================
target_col = "Attack"
feature_cols = ["shunt_voltage", "bus_voltage_V", "current_mA", "power_mW", "State"]

def prepare_features(df):
    X_raw = df[feature_cols].copy()
    y_raw = df[target_col].copy()
    
    # One-hot encode 'State'
    X = pd.get_dummies(X_raw, columns=["State"], drop_first=False)
    
    # Ensure both State columns exist
    for col in ["State_idle", "State_charging"]:
        if col not in X.columns:
            X[col] = 0
    
    X = X[["shunt_voltage", "bus_voltage_V", "current_mA", "power_mW", "State_idle", "State_charging"]]
    return X, y_raw

# Non-normalized
X_train_raw, y_train_raw = prepare_features(df_train)
X_test_raw,  y_test_raw  = prepare_features(df_test)

# Normalized
X_train_norm, _ = prepare_features(df_train_norm)
X_test_norm,  _ = prepare_features(df_test_norm)

# Encode target labels (same encoder for both experiments)
le = LabelEncoder()
y_train = le.fit_transform(y_train_raw)
y_test  = le.transform(y_test_raw)

print("\nEncoded classes:", dict(zip(le.classes_, range(len(le.classes_)))))
print("Feature columns:", X_train_raw.columns.tolist())



Encoded classes: {'Backdoor': 0, 'none': 1, 'syn-flood': 2}
Feature columns: ['shunt_voltage', 'bus_voltage_V', 'current_mA', 'power_mW', 'State_idle', 'State_charging']


In [9]:
# =============================================================================
# 4. SCALE / NORMALIZE NUMERIC FEATURES
# =============================================================================
numeric_features = ["shunt_voltage", "bus_voltage_V", "current_mA", "power_mW"]

# ---- 4a. StandardScaler on raw data ----
scaler_std = StandardScaler()
X_train_std = X_train_raw.copy()
X_test_std  = X_test_raw.copy()

X_train_std[numeric_features] = scaler_std.fit_transform(X_train_raw[numeric_features])
X_test_std[numeric_features]  = scaler_std.transform(X_test_raw[numeric_features])

X_train_std_np = X_train_std.to_numpy()
X_test_std_np  = X_test_std.to_numpy()

# ---- 4b. Use already MinMax-normalized data (no additional scaling needed) ----
# The numeric columns in df_train_norm / df_test_norm are already in [0, 1].
# We just reuse X_train_norm / X_test_norm as-is.
X_train_norm_np = X_train_norm.to_numpy()
X_test_norm_np  = X_test_norm.to_numpy()

In [10]:
# =============================================================================
# 5. TRAIN TWO RF MODELS (SAME HYPERPARAMETERS)
# =============================================================================
rf_kwargs = dict(
    n_estimators=100,
    random_state=42,
    oob_score=True,
    n_jobs=-1
)

# 5a. RF on StandardScaler-scaled raw data
rf_std = RandomForestClassifier(**rf_kwargs)
rf_std.fit(X_train_std_np, y_train)

# 5b. RF on MinMax-normalized data
rf_norm = RandomForestClassifier(**rf_kwargs)
rf_norm.fit(X_train_norm_np, y_train)

print("\n=== Models trained ===")
print(f"RF (StandardScaler) OOB score: {rf_std.oob_score_:.6f}")
print(f"RF (MinMax [0,1])   OOB score: {rf_norm.oob_score_:.6f}")


=== Models trained ===
RF (StandardScaler) OOB score: 0.868834
RF (MinMax [0,1])   OOB score: 0.869029


In [11]:
# =============================================================================
# 6. EVALUATE BOTH MODELS
# =============================================================================
def evaluate_rf(model, X_test, y_test, name):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)
    
    macro_f1 = f1_score(y_test, y_pred, average="macro")
    recon_err = mean_squared_error(y_test, y_pred)
    test_loss = log_loss(y_test, y_prob)
    
    val_loss = None
    if hasattr(model, "oob_decision_function_") and model.oob_decision_function_ is not None:
        val_loss = log_loss(y_train, model.oob_decision_function_)
    
    roc_auc = roc_auc_score(y_test, y_prob, multi_class="ovr", average="macro")
    
    print(f"\n=== {name} ===")
    print(f"  Macro_f1:        {macro_f1:.6f}")
    print(f"  Recon_Error:     {recon_err:.6f}")
    print(f"  test_loss:       {test_loss:.6f}")
    if val_loss is not None:
        print(f"  validation_loss: {val_loss:.6f}")
    else:
        print("  validation_loss: Not available")
    print(f"  ROC_AUC:         {roc_auc:.6f}")
    
    return {
        "name": name,
        "macro_f1": macro_f1,
        "recon_error": recon_err,
        "test_loss": test_loss,
        "val_loss": val_loss,
        "roc_auc": roc_auc
    }

metrics_std  = evaluate_rf(rf_std,  X_test_std_np,  y_test, "RF on StandardScaler-scaled data")
metrics_norm = evaluate_rf(rf_norm, X_test_norm_np, y_test, "RF on MinMax-normalized data [0,1]")


=== RF on StandardScaler-scaled data ===
  Macro_f1:        0.855424
  Recon_Error:     0.126384
  test_loss:       0.278408
  validation_loss: 0.313319
  ROC_AUC:         0.959268

=== RF on MinMax-normalized data [0,1] ===
  Macro_f1:        0.854333
  Recon_Error:     0.127219
  test_loss:       0.276135
  validation_loss: 0.313449
  ROC_AUC:         0.959259
